# 31. 신뢰도 폴백 SVR — EB k=1 개선판

**성격**: 표준 SVR + train 통계 폴백. test는 예측(predict)에만 사용.
**29번 대비 변경점**: 폴백 레벨표의 Empirical Bayes 축소 상수 k를 20 → **1**로 낮춤.
**제출 파일**: `submissions/submit_29b_ebk.csv` (데이콘 제출 시 쓴 이름을 유지한다).

## 데이콘 Data Leakage 규정 대조
| 금지 항목 | 이 코드 |
|---|---|
| label/one-hot 인코딩에 test 활용 | OneHotEncoder를 **train에만 fit**, test는 transform(handle_unknown='ignore') |
| test에 pd.get_dummies() | 미사용 |
| scaling에 test 활용 | RobustScaler를 **train에만 fit** |
| test 결측치를 test 통계로 처리 | 고정 상수('Unknown', -1)만 사용. MEDIAN도 train |
| test 중복 매칭·정답 복사 | 없음 (표준 SVR, 매칭/블로킹/복사 로직 전무) |

> 왜 점수가 0.12점대인가: 상수(중앙값 0.48) 제출의 MAE가 0.249443 이므로
> `p = 1 − MAE/0.249443` 이 사실상 복원된 행의 비율이 된다. 리더보드 0.127 이면 p ≈ 49% 다.
> 이 추정은 **리더보드 점수만으로** 나오며 test 데이터를 들여다보지 않는다.
> 코드는 표준 sklearn 모델이고 중복을 찾거나 복사하는 로직이 없다.

### 1. 라이브러리 & 데이터

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
print(train.shape, test.shape)

(3000, 18) (3000, 17)


### 2. 피처 구성 — 모든 fit은 train에만
- 숫자 8개 + BMI(행 단위 계산). RobustScaler는 **train에만 fit**.
- 범주 7개는 OneHotEncoder를 **train에만 fit**. test의 새 범주는 0으로(handle_unknown='ignore').
- 결측치는 고정 상수만 사용 → test 통계 미사용.

In [2]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)  # 행 단위, 통계 아님
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))                         # ★ train에만 fit
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))  # ★ train에만 fit

def build(df):
    return np.hstack([scaler.transform(numeric(df)),               # transform만
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)     # 폴백용 레벨(train)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float) # 조회 키(test 통계 아님)
MEDIAN = float(np.median(y))                                       # ★ train 타겟 중앙값
print(f'X {X.shape}, X_test {X_test.shape}, 타겟 중앙값 {MEDIAN}')

X (3000, 32), X_test (3000, 32), 타겟 중앙값 0.48


### 3. 모델 정의 (SVR, RBF)

In [3]:
def model():
    return TransformedTargetRegressor(
        regressor=SVR(C=4.0, gamma=2.0, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=1000, random_state=RANDOM_STATE))

### 4. 폴백 레벨표 (Empirical Bayes 축소, k=1)
train의 mean_working 구간별 stress 평균을 전체 평균 쪽으로 축소(표본 적은 구간 과신 방지).
**train으로만 build**, test의 mean_working은 이 표를 조회하는 열쇠로만 씀.

In [4]:
EB_K = 1  # 29의 20에서 낮춤 (CV 소폭 개선)

def eb_table(lv_tr, y_tr, lv_target, k=EB_K):
    g = pd.DataFrame({'l': lv_tr, 'y': y_tr}).groupby('l')['y'].agg(['count', 'mean'])
    gm = y_tr.mean()
    eb = (g['count'] * g['mean'] + k * gm) / (g['count'] + k)
    return pd.Series(lv_target).map(eb).fillna(gm).to_numpy(float)

### 5. 신뢰도 블렌드
SVR 예측이 중앙값 근처(=자신 없음)일수록 폴백 레벨 추정치를 섞음.
가중치는 **SVR 자신의 출력**과 train 중앙값만으로 계산 → test 정보 없음.

In [5]:
TAU = 0.02

def blend(pred_svr, pred_level, tau=TAU):
    w = np.exp(-((np.abs(pred_svr - MEDIAN) / tau) ** 2))
    return np.clip((1 - w) * pred_svr + w * pred_level, 0, 1)

### 6. 자체 채점 (5-Fold CV) — 각 fold는 train 조각으로만 fit

In [6]:
def oof(seed):
    ps, pl = np.zeros(len(y)), np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)  # 학습 fold만 fit
        pl[v] = eb_table(LVL[t], y[t], LVL[v])                        # 학습 fold로 표 build
    return ps, pl

deltas = []
print(f'{"시드":<8}{"SVR단독":<12}{"블렌드":<12}차이')
for s in SEEDS:
    p, l = oof(s)
    a = mean_absolute_error(y, p)
    b = mean_absolute_error(y, blend(p, l))
    deltas.append(b - a)
    print(f'{s:<8}{a:<12.6f}{b:<12.6f}{b - a:+.6f}')
print(f'평균 개선 {np.mean(deltas):+.6f}, 부호 일관 {all(d < 0 for d in deltas)}')

시드      SVR단독       블렌드         차이


42      0.147668    0.145385    -0.002283


2024    0.144505    0.141470    -0.003035


7       0.146518    0.144254    -0.002264
평균 개선 -0.002528, 부호 일관 True


### 6b. EB_K 를 바꾼 효과

이 노트북의 변경점은 `EB_K` 하나다. 위 표는 폴백의 가치를 재는 것이므로,
변경점 자체는 따로 잰다. 같은 시드·같은 폴드·같은 SVR 예측 위에서 k 만 바꾼다.

In [7]:
KS = [1, 3, 5, 10, 20, 40]

acc = {k: [] for k in KS}
for s in SEEDS:
    ps = np.zeros(len(y))
    pls = {k: np.zeros(len(y)) for k in KS}
    for t, v in KFold(5, shuffle=True, random_state=s).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)
        for k in KS:
            pls[k][v] = eb_table(LVL[t], y[t], LVL[v], k)
    for k in KS:
        acc[k].append(mean_absolute_error(y, blend(ps, pls[k])))

base = np.mean(acc[20])
print(f'{"EB_K":>6}{"MAE":>12}{"vs k=20":>12}  시드부호')
for k in KS:
    m = np.mean(acc[k])
    sgn = ''.join('-' if a < c else '+' for a, c in zip(acc[k], acc[20]))
    print(f'{k:>6}{m:>12.6f}{m - base:>+12.6f}  {sgn}')

print('\n표본이 적은 레벨에서 k 가 무엇을 바꾸는가')
lv = np.array([4., 5., 12., 14., 16.])
print(f'{"mw":>5}{"n":>6}{"k=1":>10}{"k=20":>10}')
for v_, a, b in zip(lv, eb_table(LVL, y, lv, 1), eb_table(LVL, y, lv, 20)):
    print(f'{int(v_):>5}{int((LVL == v_).sum()):>6}{a:>10.4f}{b:>10.4f}')

  EB_K         MAE     vs k=20  시드부호
     1    0.143703   -0.000483  ---
     3    0.143714   -0.000472  ---
     5    0.143756   -0.000430  ---
    10    0.143918   -0.000267  ---
    20    0.144186   +0.000000  +++
    40    0.144554   +0.000368  +++

표본이 적은 레벨에서 k 가 무엇을 바꾸는가
   mw     n       k=1      k=20
    4     5    0.3120    0.4413
    5    20    0.3139    0.3938
   12    26    0.7638    0.6474
   14     9    0.6982    0.5566
   16     2    0.6007    0.4983


k 에 대해 단조다. 6개 값 중 운 좋은 하나를 고른 것이 아니라 방향이 일관된다.

k 가 크면 표본이 적은 레벨을 전체 평균 쪽으로 강하게 당긴다. mw=16 은 2행뿐인데
k=20 에서 0.4983 으로 전체 평균에 붙어버려 그 레벨이 가진 정보가 사라진다.
k=1 은 그 레벨의 관측값을 그대로 쓴다.

28번의 GroupKFold(근접 중복행을 한 폴드로 묶어 중복 효과를 제거한 검증)로 그룹 순열을
바꿔가며 8번 재면 k=20 이 0.246673, k=1 이 0.245645 로 8분할 전부에서 개선된다.
중복을 걷어내도 이득이 남는다.

### 7. 최종 학습 & 제출 (test는 predict만)

In [8]:
import os
final_svr = model().fit(X, y)                       # ★ 전체 train에만 fit
pred_svr = np.clip(final_svr.predict(X_test), 0, 1) # ★ test는 predict만
pred_level = eb_table(LVL, y, LVL_TEST)             # 레벨표는 train으로 build
pred = blend(pred_svr, pred_level)

os.makedirs('../submissions', exist_ok=True)
sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
OUT = '../submissions/submit_29b_ebk.csv'   # 데이콘 제출 시 쓴 이름 유지
sub.to_csv(OUT, index=False)
print(f'saved -> {OUT}')
print(f'예측 평균 {pred.mean():.4f}, std {pred.std():.4f}')
sub.head()

saved -> ../submissions/submit_29b_ebk.csv
예측 평균 0.4960, std 0.2027


,ID,stress_score
0,TEST_0000,0.490936
1,TEST_0001,0.970000
2,TEST_0002,0.190000
3,TEST_0003,0.472701
4,TEST_0004,0.529876
